# 🎯 [PRC01_M03] 방위사업 계약 필수 규정 준수 여부 자동 판별 1차 Rule 엔진 구현
### 실습 솔루션 & 정답 해설 (Google Colab / Jupyter)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/PRC01/prc01_m03_solution.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 조달·계약·국산화 (Level 3) |
| **미션 ID** | PRC01_M03 |
| **미션 주제** | 방위사업 계약 필수 규정 준수 여부 자동 판별 1차 Rule 엔진 구현 |
| **소요 시간** | 약 2.5시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 목표
조달계획안과 계약서는 지체상금률, 선금, 보증, 하도급, 필수 조항 같은 항목이 규정에 맞는지 사람이 한 건씩 읽고 대조해야 해서 검토가 늦어지고 불공정 조항이나 납기 지체 위험 문구를 놓치기 쉽다. 본 프로젝트는 조달계획안과 계약서의 오류와 위험을 검증하는 도구를 만드는 과제로, 조달 업무의 병목과 위험 신호 기준, 조항 수정 대안, 계약 의뢰 근거와 변경 이력, 업무 효과 측정까지 한 흐름으로 구현한다. 교육용 가상 계약 자료로 계약 데이터 스키마 정의, 서식별 자료 정제와 조항 정규화, 룰 기반 규정 준수 판별, 자연어처리 기반 독소·납기 위험 문구 식별, 위험 점수와 신뢰도 엔진, 조항 하이라이트 대시보드와 오류 처리, 계약관·법무 검토 HITL 워크플로우, 1,000건 일괄 스크리닝 부하 시험, AAR 룰셋 재보정, 통합 패키지까지 단계별로 만든다. 최종 목표는 위험 계약을 근거와 함께 먼저 보여 주어 검토 시간을 줄이되, 규정 위반 여부의 판단과 계약 승인은 계약관·법무·승인권자가 하도록 지원하는 도구를 완성하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

---

### 📁 프로젝트 표준 산출물 디렉터리 체계 (`docs/`, `src/`, `dist/`) 안내
> **💡 [실무 프로토타입 업무 도구 구축 가이드]**
> 본 프로젝트는 단순 일회성 데이터 분석 실습이 아닌, 실제 군 부서(군수사령부, 조달본부, 계약관리과 등)에 배포 가능한 **업무 자동화 소프트웨어 도구(Tool)**를 단계별로 조립·완성하는 과정입니다. 실습 중 산출물(Deliverables)로 요구되는 파일들은 아래 표준 디렉터리에 모듈별로 저장·누적됩니다.
> - **`docs/` (국방 행정 및 기술 명세서):** 업무 정의서(`prc01_as_is_to_be_spec.md`), 스키마 명세서(`schema_definition.md`), UAT 검증 보고서(`prc01_uat_feedback_report.md`), 7개 장 운영 매뉴얼(`docs/prc01_final_manual.md`)
> - **`src/` (업무 도구 소스코드):** 데이터 파서(`connectors/`), 규정 준수 룰 엔진(`engine/`), NLP 지능 모듈(`ai/`), 대시보드 UI(`ui/`), 결함 검증기(`utils/`), HITL 워크플로우(`workflow/`), 배치 러너(`pipeline/`)
> - **`dist/` (배포 패키지):** 원클릭 자동 설치 스크립트(`dist/prc01_package/setup.sh`), 설치 가이드(`README.md`)
> *(주피터 노트북 환경에서는 각 미션 셀 검증 후 `%%writefile src/...` 매직 명령어를 활용하여 해당 모듈 파일로 자동 추출할 수 있습니다.)*

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 03. 방위사업 계약 필수 규정 준수 여부 자동 판별 1차 Rule 엔진 구현
**소요시간**: 약 2.5시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

방위사업 계약 필수 규정 준수 여부 판별 Rule 엔진을 구현한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **방위사업 계약 필수 규정(가상 지침)의 준수 여부를 판별하는 룰 15종 이상을 YAML 설정으로 코딩하고 계약별 1차 판별 등급을 산출한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **지체상금률 하한, 선금 한도와 보증, 수의계약 한도, 이행보증, 하자보증, 해지·보안·지식재산 조항 유무, 하도급 비율, 검사 기간, 납품 기한 논리, 물가변동 조항, 장기 계약, 불공정 키워드 등 16종 룰을 가중치 합으로 고위험·주의·관심 등급에 연결 수행하고 검증하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/engine/prc01_rule_engine.py` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 군 규정 및 지침 조항과의 정확한 일치성

#### 미션 PASS 조건

- 정량 기준: 업무 검증 룰 15종 이상 코딩, 룰 처리 지연시간 50ms 이하


- 진행 기준: 룰 엔진 단위 테스트 100% 통과


- 검증 방법: 경계 조건(Boundary condition)에 대한 룰 판정 검증


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""계약 필수 규정 준수 여부를 판별하는 16종 룰(YAML 설정)을 코딩해 계약별 1차 판별 등급을 산출하고, 경계값 단위 시험 32건, 룰 처리 지연시간, 룰별 근거 조항 연결, 등급별 실제 위반율·재현율, 규정 변경 시 설정만 바꾸는 확장성을 확인한다."""
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 조달계획안·계약서 생성기 (실제 계약·업체·법령 조문 없음)
# 계약 속성 16개 + 조항 문장 6개를 만들고, 속성 위반·독소 조항·납기 위험 문구가 섞인 '규정 위반 여부'를 함께 만든다.
# ---------------------------------------------------------------------------
TYPES = ["물품", "용역", "공사"]
METHODS = ["일반경쟁", "제한경쟁", "지명경쟁", "수의계약"]
CLASSES = ["정상", "독소", "납기위험"]
OBJ = ["납품 물품", "용역 결과물", "시공 목적물", "정비 성과물"]

N_TPL = [
    "계약상대자가 납기를 지체한 경우 지체일수 1일마다 계약금액의 {p}/1000에 해당하는 지체상금을 부과한다.",
    "지체상금은 불가항력 사유가 확인된 경우에 한하여 발주기관의 승인을 받아 면제할 수 있다.",
    "납기 연장은 발주기관이 불가항력 사유를 확인하고 서면으로 승인한 경우에만 인정한다.",
    "발주기관은 납품된 {o}을 {d}일 이내에 검사하고 합격한 후에 대금을 지급한다.",
    "계약상대자는 하도급을 하려는 경우 사전에 발주기관의 서면 승인을 받아야 한다.",
    "어느 한쪽이 계약을 해지하려면 사유를 명시해 {d}일 전에 서면으로 통지해야 한다.",
    "{o}에 하자가 발생하면 계약상대자는 보증 기간 안에 무상으로 보수하거나 교체한다.",
    "계약 이행 중 알게 된 비공개 사항은 제3자에게 누설하지 않으며 위반 시 책임을 진다.",
    "{o}에 대한 지식재산권의 귀속과 사용 범위는 별도 합의서에 따라 정한다.",
    "대금은 검사 합격 후 청구서를 접수한 날부터 {d}일 이내에 지급한다.",
    "계약금액의 조정은 물가변동 기준에 따라 양 당사자가 서면으로 합의한 경우에만 한다.",
    "계약상대자는 이행보증서를 계약 체결 시 제출하며 보증금은 계약 이행 완료 후 반환한다.",
    "납품 일정은 계약서 별표의 일자를 따르며 변경 시 서면 합의가 필요하다.",
    "지체상금은 면제 사유가 없는 한 부과하며 이의는 정해진 절차로 제기할 수 있다.",
    "검사에 합격하지 못한 {o}은 계약상대자가 교체하며 그 비용은 계약상대자가 부담한다.",
    "계약상대자의 귀책 사유로 이행이 불가능하면 발주기관은 서면으로 통지하고 계약을 해지할 수 있다.",
    "지연이 예상되면 계약상대자는 사유와 만회 계획을 발주기관에 서면으로 보고한다.",
    "보증 기간 중 발생한 결함은 계약상대자가 책임지고 보수한다.",
    "분쟁은 먼저 협의로 해결하고 협의가 되지 않으면 관할 법원의 판단에 따른다.",
]
T_TPL = [
    "계약상대자의 지체상금은 전액 면제하며 발주기관은 이에 대해 이의를 제기하지 못한다.",
    "납품 즉시 검사에 합격한 것으로 보며 발주기관은 이후 검사를 요구하지 않는다.",
    "계약상대자는 언제든지 일방적으로 계약을 해지할 수 있으며 그에 따른 손해는 발주기관이 부담한다.",
    "{o}의 추가 비용은 사유를 불문하고 발주기관이 전액 부담하며 별도 협의 없이 청구할 수 있다.",
    "{o}에 관한 지식재산권은 모두 계약상대자에게 귀속되고 발주기관은 사용할 수 없다.",
    "하도급은 신고 없이 제한 없이 허용하며 발주기관은 하도급 내용을 확인하지 않는다.",
    "대금은 검사 전에 전액 선지급하며 발주기관은 반환 청구권을 포기한다.",
    "계약 이행 중 알게 된 사항의 누설로 인한 책임을 계약상대자는 지지 않는다.",
    "하자 보수 책임은 인도 후 즉시 소멸하며 발주기관은 보수를 요구할 수 없다.",
    "발주기관은 계약상대자가 청구하는 금액을 거절할 수 없으며 거절하면 위약금을 지급한다.",
    "분쟁이 생기면 계약상대자가 정하는 중재 기관에서만 해결하고 발주기관은 따른다.",
    "발주기관의 검사는 생략하고 계약상대자의 시험 결과서로 대신한다.",
    "견적 금액은 사후 정산 없이 그대로 확정하고 증빙 자료는 받지 않는다.",
]
D_TPL = [
    "납기는 자재 수급 사정에 따라 계약상대자가 연장할 수 있다.",
    "납품 일정은 가능한 한 지키도록 노력하며 지연되어도 사유를 보고하지 않는다.",
    "공급 지연이나 선행 업체 사정도 불가항력으로 보아 납기를 자동 연장한다.",
    "착수일은 선행 업체의 납품이 끝난 뒤 별도로 정하며 완료 기한은 정하지 않는다.",
    "납기일 이후 1회에 한해 기한 없이 자동 연장되는 것으로 한다.",
    "납기는 계약상대자의 생산 일정이 확정되는 대로 통보한다.",
    "납품 시점은 계약상대자 사정이 허락하는 때로 하고 날짜는 따로 정하지 않는다.",
    "자재가 늦게 들어오면 완료 일정은 그만큼 자동으로 늦춰진다.",
    "기한 내에 인도하지 못해도 계약상대자에게 책임을 묻지 않는다.",
    "일정이 늦어져도 계약상대자는 새 계획서를 낼 필요가 없다.",
]
# 학습에 쓰지 않는 '처음 보는 표현'(모델이 문장 틀을 외웠는지 확인하는 용도)
HOLD = {
    "정상": ["검사에 불합격한 경우 계약상대자는 지정 기한 안에 재납품하며 재검사 비용은 상대자가 부담한다.",
           "계약상대자가 법령을 위반하여 계약을 이행하지 못하면 발주기관은 서면 통지 후 계약을 해지할 수 있다.",
           "납기가 지연될 우려가 있으면 상대자는 즉시 사유와 만회 계획을 서면으로 보고해야 한다.",
           "하자담보 책임 기간은 인수일부터 기산하며 기간 중 발생한 결함은 상대자가 책임진다."],
    "독소": ["발주기관은 계약상대자가 요구하는 대금을 이유 없이 거절하지 못하며 거절 시 위약금을 낸다.",
           "계약 내용에 관한 분쟁은 계약상대자가 지정하는 기관에서만 다루며 발주기관은 이에 따른다.",
           "발주기관의 검수 권한은 포기하고 계약상대자의 자체 시험 성적서로 갈음한다.",
           "계약상대자가 제출한 견적은 사후 정산 없이 확정 단가로 하며 증빙은 요구하지 않는다."],
    "납기위험": ["납품 시기는 상대자의 형편이 허락하는 시점으로 하고 별도 날짜는 정하지 않는다.",
             "원자재 확보가 늦어지면 완료 일자는 자동으로 뒤로 미뤄지는 것으로 본다.",
             "인도 기한은 참고용이며 지키지 못해도 책임을 묻지 않는다.",
             "일정 지연이 있어도 상대자는 변경 계획을 제출할 의무가 없다."],
}
SUFFIX = ["", "", "", " 세부 사항은 시행 세칙에 따른다.", " 이 조항은 계약 전체에 적용한다.", " 필요한 경우 양 당사자가 협의한다.", " 관련 서식은 별지를 따른다."]
SWAP = [("발주기관", "계약담당관"), ("계약상대자", "업체")]


def make_clause(rng, cls, hold=False, idx=None):
    """조항 한 문장 생성. 표현 일부는 동의어로 바꾸고 접미 문구를 무작위로 붙인다(idx를 주면 그 번호의 문장 틀을 사용)."""
    if idx is not None:
        tpl = {"정상": N_TPL, "독소": T_TPL, "납기위험": D_TPL}[cls]
        s = tpl[idx].format(p=rng.choice(["1.0", "1.5", "2.0"]), d=int(rng.choice([7, 10, 14, 30])), o=rng.choice(OBJ))
    elif hold:
        s = HOLD[cls][rng.integers(len(HOLD[cls]))]
    else:
        tpl = {"정상": N_TPL, "독소": T_TPL, "납기위험": D_TPL}[cls]
        s = tpl[rng.integers(len(tpl))].format(p=rng.choice(["1.0", "1.5", "2.0"]), d=int(rng.choice([7, 10, 14, 30])), o=rng.choice(OBJ))
    if rng.random() < 0.3:
        a, b = SWAP[rng.integers(2)]
        s = s.replace(a, b)
    return s + SUFFIX[rng.integers(len(SUFFIX))]


def make_contracts(n=200, seed=7):
    """가상 계약 n건(속성 16개 + 조항 6개 + 규정 위반 정답)을 만든다."""
    rng = np.random.default_rng(seed)
    sl = rng.beta(1.2, 3.2, n)                                  # 계약서 작성이 허술한 정도(잠재 변수)
    ctype = rng.choice(TYPES, n, p=[.5, .3, .2])
    amount = np.clip(np.exp(rng.normal(np.log(220), 1.1, n)), 5, 5000).round(0)
    method = np.where(rng.random(n) < .06 + .3 * sl, "수의계약", rng.choice(METHODS[:3], n, p=[.5, .3, .2]))
    period = np.array([rng.integers(60, 300) if t == "물품" else rng.integers(90, 400) if t == "용역" else rng.integers(180, 720) for t in ctype])
    delivery = (period * rng.uniform(.5, 1.0, n)).astype(int)
    bad_logic = rng.random(n) < .015 * (1 + 3 * sl)
    delivery = np.where(bad_logic, period + rng.integers(10, 60, n), delivery)
    pen = np.where(rng.random(n) < .22 * sl, rng.choice([.3, .5, .8], n), rng.choice([1.0, 1.5, 2.0, 2.5], n))
    adv = np.where(rng.random(n) < .22 * sl, rng.choice([50, 60, 70], n), rng.choice([0, 10, 20, 30, 40], n))
    bond = np.where(rng.random(n) < .25 * sl + .5 * (adv > 40), rng.choice([0, 5], n), rng.choice([10, 10, 15, 20], n))
    war = np.array([rng.choice([3, 6]) if (rng.random() < .25 * s and t == "물품") else rng.choice([12, 24, 36]) if t == "물품" else rng.choice([0, 6, 12]) if t == "용역" else rng.choice([24, 36, 60]) for t, s in zip(ctype, sl)])
    insp = np.where(rng.random(n) < .3 * sl, rng.choice([21, 30], n), rng.choice([5, 7, 10, 14], n))
    sub = np.where(rng.random(n) < .15 * sl, rng.choice([60, 70, 80], n), rng.choice([0, 10, 20, 30, 40, 50], n))
    padj = rng.random(n) < .75 - .4 * sl
    term = rng.random(n) > .015 * (1 + 6 * sl)
    sec = rng.random(n) > .015 * (1 + 6 * sl)
    ip = rng.random(n) > .03 * (1 + 4 * sl)
    clauses, cls_true = [], []
    for i in range(n):
        pt, pd_ = .01 + .08 * sl[i], .02 + .08 * sl[i]
        cl = [rng.choice(CLASSES, p=[1 - pt - pd_, pt, pd_]) for _ in range(6)]
        cls_true.append(cl); clauses.append([make_clause(rng, c) for c in cl])
    df = pd.DataFrame({"contract_id": [f"C-{i + 1:04d}" for i in range(n)], "ctype": ctype, "method": method, "amount_m": amount,
                       "period_days": period, "delivery_days": delivery, "penalty_permil": pen, "advance_pct": adv, "perf_bond_pct": bond,
                       "warranty_months": war, "inspection_days": insp, "subcontract_pct": sub, "price_adjust": padj,
                       "termination": term, "security": sec, "ip_clause": ip})
    hard = ((pen < 1.0) | ((adv > 40) & (bond < 10)) | ((amount >= 300) & (method == "수의계약")) | ((ctype == "물품") & (war < 12))
            | ~term | ~sec | (sub > 50) | (delivery > period))
    toxic = np.array([c.count("독소") for c in cls_true])
    delay = np.array([c.count("납기위험") for c in cls_true])
    split = (method == "수의계약") & (amount >= 250) & (amount < 300) & (rng.random(n) < .6)      # 계약 쪼개기 의심(임계값 바로 아래)
    df["clauses"], df["clause_cls"] = clauses, cls_true
    df["n_toxic"], df["n_delay"] = toxic, delay
    viol = hard | (toxic > 0) | split | (delay >= 2)
    waived = hard & (toxic == 0) & (rng.random(n) < .12)                  # 계약관이 예외 사유를 인정해 위반으로 보지 않은 건(자료의 현실적 잡음)
    hidden = ~viol & (rng.random(n) < .03 + .08 * sl)                       # 속성·문구에 드러나지 않는 위반(별첨 합의 등)
    df["violation"] = (viol & ~waived) | hidden
    df["vtype"] = np.where(hidden, "기타", np.where(hard, "속성위반", np.where(toxic > 0, "독소조항", np.where(split | (delay >= 2), "기타", "-"))))
    return df

# ---- 계약 규정 준수 판별 룰 설정(YAML 형식 문자열: 규정이 바뀌면 코드가 아니라 이 설정만 고친다) ----
RULES_YAML = """
- id: R01
  name: 지체상금률 1.0/1000 미만
  field: penalty_permil
  op: "<"
  value: 1.0
  weight: 3
- id: R02
  name: 선금 비율 40% 초과
  field: advance_pct
  op: ">"
  value: 40
  weight: 2
- id: R03
  name: 선금 40% 초과인데 이행보증 10% 미만
  field: adv_no_bond
  op: "=="
  value: true
  weight: 3
- id: R04
  name: 300백만원 이상 수의계약
  field: sole_amount
  op: ">="
  value: 300
  weight: 3
- id: R05
  name: 500백만원 이상 이행보증 10% 미만
  field: bond_gap
  op: "=="
  value: true
  weight: 2
- id: R06
  name: 물품 하자보증 12개월 미만
  field: warranty_gap
  op: "=="
  value: true
  weight: 2
- id: R07
  name: 해지 조항 없음
  field: termination
  op: "=="
  value: false
  weight: 2
- id: R08
  name: 보안 조항 없음
  field: security
  op: "=="
  value: false
  weight: 3
- id: R09
  name: 지식재산 조항 없음(물품·용역)
  field: ip_missing
  op: "=="
  value: true
  weight: 1
- id: R10
  name: 하도급 비율 50% 초과
  field: subcontract_pct
  op: ">"
  value: 50
  weight: 3
- id: R11
  name: 검사 기간 14일 초과
  field: inspection_days
  op: ">"
  value: 14
  weight: 1
- id: R12
  name: 납품 기한이 계약 기간 초과
  field: logic_err
  op: "=="
  value: true
  weight: 3
- id: R13
  name: 180일 이상 계약에 물가변동 조항 없음
  field: no_padj_long
  op: "=="
  value: true
  weight: 1
- id: R14
  name: 1000백만원 이상 비경쟁 계약
  field: noncomp_big
  op: "=="
  value: true
  weight: 2
- id: R15
  name: 계약 기간 365일 초과
  field: period_days
  op: ">"
  value: 365
  weight: 1
- id: R16
  name: 불공정·납기 위험 키워드 포함
  field: kw_hits
  op: ">="
  value: 1
  weight: 3
"""
GRADE_CUT = {"고위험": 6, "주의": 3}          # 가중치 합 기준 1차 판별 등급(필요 시 설정만 바꾼다)
# [교육용가상자료] 불공정 조항·납기 지체 위험 문구 사전(실제 규정 조문이 아님)
KEYWORDS = {"독소": ["면제하며", "일방적으로", "포기", "지지 않는다", "즉시 소멸", "제한 없이", "신고 없이", "전액 부담", "별도 협의 없이", "검사 전에", "검사에 합격한 것으로", "사용할 수 없다", "증빙은 요구하지 않는다", "지정하는 기관"],
            "납기위험": ["자동 연장", "기한 없이", "정하지 않는다", "노력하며", "연장할 수 있다", "통보한다", "책임을 묻지 않는다", "의무가 없다"]}


def parse_rules(text):
    """간이 YAML 파서(목록 + key: value). 숫자·true/false·따옴표 문자열을 해석한다."""
    rules, cur = [], None
    for line in text.strip().splitlines():
        if not line.strip():
            continue
        s = line.strip()
        if s.startswith("- "):
            cur = {}; rules.append(cur); s = s[2:]
        k, v = [x.strip() for x in s.split(":", 1)]
        v = v.strip('"')
        if v in ("true", "false"):
            v = v == "true"
        else:
            try:
                v = float(v) if "." in v else int(v)
            except ValueError:
                pass
        cur[k] = v
    return rules


OPS = {"==": lambda a, b: a == b, "!=": lambda a, b: a != b, ">": lambda a, b: a > b, ">=": lambda a, b: a >= b,
       "<": lambda a, b: a < b, "<=": lambda a, b: a <= b}


def kw_count(clauses):
    """조항 문장 목록에서 독소·납기 위험 키워드가 들어 있는 조항 수."""
    return int(sum(any(k in c for k in KEYWORDS["독소"] + KEYWORDS["납기위험"]) for c in clauses))


def derive(df):
    """룰에 쓰는 파생 필드(선금+보증 부족, 수의계약 고액, 논리 오류, 키워드 적중 등)를 만든다."""
    d = df.copy()
    d["adv_no_bond"] = (d["advance_pct"] > 40) & (d["perf_bond_pct"] < 10)
    d["sole_amount"] = np.where(d["method"] == "수의계약", d["amount_m"], 0.0)       # 수의계약일 때만 금액(한도는 YAML에서 조정)
    d["bond_gap"] = (d["amount_m"] >= 500) & (d["perf_bond_pct"] < 10)
    d["warranty_gap"] = (d["ctype"] == "물품") & (d["warranty_months"] < 12)
    d["ip_missing"] = (~d["ip_clause"]) & (d["ctype"] != "공사")
    d["logic_err"] = d["delivery_days"] > d["period_days"]
    d["no_padj_long"] = (d["period_days"] >= 180) & (~d["price_adjust"])
    d["noncomp_big"] = d["method"].isin(["지명경쟁", "수의계약"]) & (d["amount_m"] >= 1000)
    d["kw_hits"] = [kw_count(c) for c in d["clauses"]] if "clauses" in d else 0
    return d


def apply_rules(rec, rules):
    """레코드 1건(dict)에 룰을 적용해 (점수, 등급, 적중 룰 id 목록)을 돌려준다. 값이 없으면 해당 룰은 건너뛴다."""
    hits = []
    for r in rules:
        v = rec.get(r["field"])
        if v is None or (isinstance(v, float) and np.isnan(v)):
            continue
        if OPS[r["op"]](v, r["value"]):
            hits.append(r)
    score = sum(r["weight"] for r in hits)
    grade = "고위험" if score >= GRADE_CUT["고위험"] else ("주의" if score >= GRADE_CUT["주의"] else "관심")
    return score, grade, [r["id"] for r in hits]

# [교육용가상자료] 룰과 가상 지침 조항의 대응표(실제 법령·규정 조문이 아니라 연결 방식을 보여 주는 예시)
CLAUSE = {"R01": "가상지침 5조1항(지체상금)", "R02": "가상지침 6조1항(선금 한도)", "R03": "가상지침 6조2항(선금과 보증)", "R04": "가상지침 3조1항(수의계약 한도)",
          "R05": "가상지침 6조3항(이행보증)", "R06": "가상지침 7조1항(하자보증)", "R07": "가상지침 8조1항(계약 해지)", "R08": "가상지침 9조1항(보안 조항)",
          "R09": "가상지침 9조2항(지식재산)", "R10": "가상지침 4조1항(하도급 제한)", "R11": "가상지침 5조2항(검사 기간)", "R12": "가상지침 5조3항(납품 기한)",
          "R13": "가상지침 6조4항(물가변동)", "R14": "가상지침 3조2항(경쟁 계약 원칙)", "R15": "가상지침 3조3항(장기 계약)", "R16": "가상지침 8조2항(불공정 조항)"}

rules = parse_rules(RULES_YAML)
con = make_contracts(200, seed=7)
df = derive(con)
recs = df.to_dict("records")

# 1) 전체 스크리닝과 처리 지연시간(레코드 1건당)
apply_rules(recs[0], rules)                                  # 첫 호출 지연(예열)을 측정에서 제외
lat, res = [], []
for r in recs:
    t0 = time.perf_counter()
    out = apply_rules(r, rules)
    lat.append((time.perf_counter() - t0) * 1000)
    res.append(out)
df["score"], df["grade"], df["hits"] = [x[0] for x in res], [x[1] for x in res], [x[2] for x in res]
print(f"[룰 설정] 검증 룰 {len(rules)}종(목표 15종 이상: {'달성' if len(rules) >= 15 else '미달'}), 모든 룰에 가상 지침 조항 연결 {all(r['id'] in CLAUSE for r in rules)}")
print(f"룰 처리 지연시간(레코드 1건): 평균 {np.mean(lat):.3f}ms, 99백분위 {np.percentile(lat, 99):.3f}ms, 최대 {np.max(lat):.3f}ms (목표 50ms 이하, 99백분위 기준: {'달성' if np.percentile(lat, 99) <= 50 else '미달'})")

# 2) 경계 조건 단위 시험: 기준값 바로 아래·위에서 룰이 정확히 켜지고 꺼지는가
CLEAN = [make_clause(np.random.default_rng(i), "정상") for i in range(6)]
BASE = dict(contract_id="C-0000", ctype="물품", method="일반경쟁", amount_m=100.0, period_days=120, delivery_days=100, penalty_permil=1.5, advance_pct=20, perf_bond_pct=15,
            warranty_months=24, inspection_days=7, subcontract_pct=20, price_adjust=True, termination=True, security=True, ip_clause=True, clauses=CLEAN)
TESTS = [("R01", dict(penalty_permil=0.9), True), ("R01", dict(penalty_permil=1.0), False), ("R02", dict(advance_pct=41), True), ("R02", dict(advance_pct=40), False),
         ("R03", dict(advance_pct=50, perf_bond_pct=9), True), ("R03", dict(advance_pct=50, perf_bond_pct=10), False),
         ("R04", dict(method="수의계약", amount_m=300), True), ("R04", dict(method="수의계약", amount_m=299), False),
         ("R05", dict(amount_m=500, perf_bond_pct=9), True), ("R05", dict(amount_m=499, perf_bond_pct=9), False),
         ("R06", dict(warranty_months=11), True), ("R06", dict(warranty_months=12), False), ("R07", dict(termination=False), True), ("R07", dict(termination=True), False),
         ("R08", dict(security=False), True), ("R08", dict(security=True), False), ("R09", dict(ip_clause=False, ctype="용역"), True), ("R09", dict(ip_clause=False, ctype="공사"), False),
         ("R10", dict(subcontract_pct=51), True), ("R10", dict(subcontract_pct=50), False), ("R11", dict(inspection_days=15), True), ("R11", dict(inspection_days=14), False),
         ("R12", dict(delivery_days=121), True), ("R12", dict(delivery_days=120), False), ("R13", dict(period_days=180, delivery_days=170, price_adjust=False), True),
         ("R13", dict(period_days=179, delivery_days=170, price_adjust=False), False), ("R14", dict(method="지명경쟁", amount_m=1000), True), ("R14", dict(method="지명경쟁", amount_m=999), False),
         ("R15", dict(period_days=366, delivery_days=300), True), ("R15", dict(period_days=365, delivery_days=300), False),
         ("R16", dict(clauses=CLEAN[:5] + ["계약상대자는 언제든지 일방적으로 계약을 해지할 수 있다."]), True), ("R16", dict(clauses=CLEAN), False)]
passed = 0
for rid, chg, expect in TESTS:
    row = derive(pd.DataFrame([{**BASE, **chg}])).iloc[0].to_dict()
    got = rid in apply_rules(row, rules)[2]
    passed += got == expect
    if got != expect:
        print(f"  [FAIL] {rid} {chg} 기대 {expect} 결과 {got}")
base_hits = apply_rules(derive(pd.DataFrame([BASE])).iloc[0].to_dict(), rules)
print(f"\n[경계 조건 단위 시험] {passed}/{len(TESTS)}건 통과 ({passed / len(TESTS):.0%}), 정상 기준 계약의 적중 룰 {base_hits[2] or '없음'} -> 등급 {base_hits[1]} (목표 100% 통과: {'달성' if passed == len(TESTS) else '미달'})")

# 3) 등급 분포와 실제 위반 여부(법무·계약관 검토에서 확인된 규정 위반 정답) 대비 탐지력
print("\n[1차 판별 등급 분포]")
print(df.groupby("grade").agg(계약수=("contract_id", "size"), 실제위반율=("violation", "mean")).reindex(["고위험", "주의", "관심"]).round(2).to_string())
y = df["violation"].to_numpy()
flag = df["grade"].isin(["고위험", "주의"]).to_numpy()
tp = int((flag & y).sum())
print(f"고위험·주의 선별: 실제 위반 계약 {y.sum()}건 중 {tp}건 포착(재현율 {tp / y.sum():.0%}), 선별 {flag.sum()}건 중 정밀도 {tp / max(flag.sum(), 1):.0%}")

# 4) 룰별 적중 건수와 근거 조항
cnt = pd.Series([h for hs in df["hits"] for h in hs]).value_counts()
tbl = pd.DataFrame([{"룰": r["id"], "이름": r["name"], "가중치": r["weight"], "적중": int(cnt.get(r["id"], 0)), "근거 조항": CLAUSE[r["id"]]} for r in rules])
print("\n[룰별 적중 현황]")
print(tbl.to_string(index=False))

# 5) 고위험 상위 3건의 판정 근거(설명 가능성)
print("\n[고위험 상위 3건과 근거]")
for _, r in df.sort_values("score", ascending=False).head(3).iterrows():
    print(f"  {r['contract_id']} {r['ctype']} {r['method']} {r['amount_m']:.0f}백만원 점수 {r['score']} {r['grade']}")
    for h in r["hits"][:3]:
        print(f"     - {h} {next(x['name'] for x in rules if x['id'] == h)} / {CLAUSE[h]}")

# 6) 규정 변경 대응: 수의계약 한도를 300에서 250백만원으로 낮추는 설정 변경(코드 수정 없음)
rules2 = parse_rules(RULES_YAML.replace('op: ">="\n  value: 300', 'op: ">="\n  value: 250'))
o2 = [apply_rules(r, rules2) for r in recs]
g2 = np.array([x[1] in ("고위험", "주의") for x in o2])
n_new = int((g2 & ~flag).sum())
print(f"\n[설정 변경] 수의계약 한도 300 -> 250백만원(YAML 값만 변경): 새로 선별된 계약 {n_new}건(계약 쪼개기 의심 구간 250~299백만원), 재현율 {tp / y.sum():.0%} -> {(g2 & y).sum() / y.sum():.0%}")
rules3 = parse_rules(RULES_YAML.replace("weight: 3\n- id: R02", "weight: 3\n- id: R02").replace("op: \"<\"\n  value: 1.0", "op: \"<\"\n  value: 1.2"))
o3 = [apply_rules(r, rules3)[0] for r in recs]
print(f"[설정 변경] 룰 R01 지체상금률 기준 1.0 -> 1.2: 점수가 바뀐 계약 {sum(a != b for a, b in zip(df['score'], o3))}건 (YAML 값만 변경)")
print("처리 후 사람 검토: 룰 판별은 1차 선별이며, 최종 규정 위반 판단과 수정 요구는 계약관·법무 담당자가 결정한다.")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].bar(tbl["룰"], tbl["적중"], color="tab:blue"); ax[0].set_title("Rule hits by rule (synthetic)"); ax[0].set_ylabel("contracts")
g = df.groupby("grade")["violation"].mean().reindex(["관심", "주의", "고위험"])
ax[1].bar(["low", "watch", "high"], g.values, color=["tab:green", "gold", "tab:red"]); ax[1].set_title("Actual violation rate by screening grade"); ax[1].set_ylim(0, 1.05)
plt.tight_layout(); plt.show()


<details>
<summary><b>💡 [참고] 미션 03 예상 실행 결과 (Output)</b></summary>

```text
[룰 설정] 검증 룰 16종(목표 15종 이상: 달성), 모든 룰에 가상 지침 조항 연결 True
룰 처리 지연시간(레코드 1건): 평균 0.008ms, 99백분위 0.036ms, 최대 0.100ms (목표 50ms 이하, 99백분위 기준: 달성)

[경계 조건 단위 시험] 32/32건 통과 (100%), 정상 기준 계약의 적중 룰 없음 -> 등급 관심 (목표 100% 통과: 달성)

[1차 판별 등급 분포]
       계약수  실제위반율
grade            
고위험     22   1.00
주의      68   0.72
관심     110   0.07
고위험·주의 선별: 실제 위반 계약 79건 중 71건 포착(재현율 90%), 선별 90건 중 정밀도 79%

[룰별 적중 현황]
  룰                      이름  가중치  적중               근거 조항
R01       지체상금률 1.0/1000 미만    3  10     가상지침 5조1항(지체상금)
R02            선금 비율 40% 초과    2  11    가상지침 6조1항(선금 한도)
R03 선금 40% 초과인데 이행보증 10% 미만    3   5   가상지침 6조2항(선금과 보증)
R04          300백만원 이상 수의계약    3  10  가상지침 3조1항(수의계약 한도)
R05   500백만원 이상 이행보증 10% 미만    2   4     가상지침 6조3항(이행보증)
R06         물품 하자보증 12개월 미만    2   5     가상지침 7조1항(하자보증)
R07                해지 조항 없음    2   8    가상지침 8조1항(계약 해지)
R08                보안 조항 없음    3   5    가상지침 9조1항(보안 조항)
R09       지식재산 조항 없음(물품·용역)    1   9     가상지침 9조2항(지식재산)
R10           하도급 비율 50% 초과    3   8   가상지침 4조1항(하도급 제한)
R11            검사 기간 14일 초과    1  21    가상지침 5조2항(검사 기간)
R12         납품 기한이 계약 기간 초과    3   7    가상지침 5조3항(납품 기한)
R13  180일 이상 계약에 물가변동 조항 없음    1  38     가상지침 6조4항(물가변동)
R14       1000백만원 이상 비경쟁 계약    2   3 가상지침 3조2항(경쟁 계약 원칙)
R15           계약 기간 365일 초과    1  24    가상지침 3조3항(장기 계약)
R16        불공정·납기 위험 키워드 포함    3  61   가상지침 8조2항(불공정 조항)

[고위험 상위 3건과 근거]
  C-0108 물품 수의계약 961백만원 점수 12 고위험
     - R01 지체상금률 1.0/1000 미만 / 가상지침 5조1항(지체상금)
     - R04 300백만원 이상 수의계약 / 가상지침 3조1항(수의계약 한도)
     - R12 납품 기한이 계약 기간 초과 / 가상지침 5조3항(납품 기한)
  C-0062 용역 수의계약 1109백만원 점수 11 고위험
     - R01 지체상금률 1.0/1000 미만 / 가상지침 5조1항(지체상금)
     - R04 300백만원 이상 수의계약 / 가상지침 3조1항(수의계약 한도)
... (이하 생략)
```
</details>
